In [1]:
stm_graph_path = "."

# Import required libraries
import sys
sys.path.append(stm_graph_path)
import stm_graph
import os
import pandas as pd
import numpy as np
from datetime import timedelta
import matplotlib.pyplot as plt
import contextily as ctx

# Define the data and output directories
DATA_DIR = "nyc_crash_311"
DATASET = "Motor_Vehicle_Collisions_-_Crashes_20261002.csv"
OUTPUT_DIR = "nyc_crash_311/stm_graph/nyc/crash"

# Define geographic boundaries for NYC
NYC_BOUNDS = {
    "min_lat": 40.4774,  # Southern boundary
    "max_lat": 40.9176,  # Northern boundary
    "min_lon": -74.2591,  # Western boundary
    "max_lon": -73.7004,  # Eastern boundary
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
# Step 1: Read the raw data
print(f"Reading raw crash data from {DATASET}")
raw_df = pd.read_csv(DATASET, low_memory=False)

# Display basic information about the dataset
print(f"Dataset shape: {raw_df.shape}")
print("\nColumn names:")
print(raw_df.columns.tolist())
print("\nSample data:")
raw_df.head()

Reading raw crash data from Motor_Vehicle_Collisions_-_Crashes_20261002.csv
Dataset shape: (636910, 29)

Column names:
['CRASH DATE', 'CRASH TIME', 'BOROUGH', 'ZIP CODE', 'LATITUDE', 'LONGITUDE', 'LOCATION', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME', 'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED', 'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED', 'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED', 'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED', 'CONTRIBUTING FACTOR VEHICLE 1', 'CONTRIBUTING FACTOR VEHICLE 2', 'CONTRIBUTING FACTOR VEHICLE 3', 'CONTRIBUTING FACTOR VEHICLE 4', 'CONTRIBUTING FACTOR VEHICLE 5', 'COLLISION_ID', 'VEHICLE TYPE CODE 1', 'VEHICLE TYPE CODE 2', 'VEHICLE TYPE CODE 3', 'VEHICLE TYPE CODE 4', 'VEHICLE TYPE CODE 5']

Sample data:


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,01/02/2020,0:00,NaN,NaN,NaN,NaN,NaN,CROSS ISLAND PARKWAY,NaN,NaN,...,NaN,NaN,NaN,NaN,4267700,Sedan,NaN,NaN,NaN,NaN
1,01/02/2020,12:57,NaN,NaN,NaN,NaN,NaN,W 57 & 8th Ave,W 57,NaN,...,Unspecified,NaN,NaN,NaN,4268255,Taxi,Pick-up Truck,NaN,NaN,NaN
2,01/02/2020,20:24,NaN,NaN,NaN,NaN,NaN,HUTCHINSON RIVER PARKWAY,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4268404,Station Wagon/Sport Utility Vehicle,Sedan,NaN,NaN,NaN
3,01/02/2020,18:23,NaN,NaN,NaN,NaN,NaN,VAN WYCK EXPWY,NaN,NaN,...,Unspecified,Unspecified,Unspecified,NaN,4268152,Sedan,Sedan,Sedan,Sedan,NaN
4,01/02/2020,19:00,NaN,NaN,40.699955,-73.98682,"(40.699955, -73.98682)",JAY STREET,NaN,NaN,...,Passing or Lane Usage Improper,NaN,NaN,NaN,4268117,Sedan,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN


In [3]:
# Step 2: Combine date and time to create a datetime column
print("Combining date and time columns...")
raw_df["created_time"] = pd.to_datetime(
    raw_df["CRASH DATE"] + " " + raw_df["CRASH TIME"], errors="coerce"
)

# Create a temporary CSV file with the combined datetime column
temp_csv = os.path.join(DATA_DIR, "temp_combined_crash.csv")
raw_df.to_csv(temp_csv, index=False)

print(f"Time range of data: {raw_df['created_time'].min()} to {raw_df['created_time'].max()}")
raw_df[["CRASH DATE", "CRASH TIME", "created_time"]].head()

Combining date and time columns...
Time range of data: 2020-01-02 00:00:00 to 2026-06-11 13:08:00


,CRASH DATE,CRASH TIME,created_time
0,01/02/2020,0:00,2020-01-02 00:00:00
1,01/02/2020,12:57,2020-01-02 12:57:00
2,01/02/2020,20:24,2020-01-02 20:24:00
3,01/02/2020,18:23,2020-01-02 18:23:00
4,01/02/2020,19:00,2020-01-02 19:00:00


In [4]:
# Process with STM-Graph
print("Processing with STM-Graph...")
gdf_crash = stm_graph.preprocess_dataset(
    data_path=DATA_DIR,
    dataset="temp_combined_crash.csv",
    time_col="created_time",
    lat_col="LATITUDE",
    lng_col="LONGITUDE",
    column_mapping={
        "created_time": "created_time",
        "BOROUGH": "borough",
        "ZIP CODE": "zip_code",
        "LATITUDE": "latitude",
        "LONGITUDE": "longitude",
        "LOCATION": "location",
        "COLLISION_ID": "collision_id",
        "NUMBER OF PERSONS INJURED": "injured_count",
        "NUMBER OF PERSONS KILLED": "killed_count",
    },
    #filter_dates=(None, "2019-12-31 23:59:59"),
    testing_mode=True,
    test_bounds=NYC_BOUNDS,
    visualize=True,
    fig_format="png",
    output_dir=OUTPUT_DIR,
    show_background_map=True,
)

print(f"Processed dataset shape: {gdf_crash.shape}")
print(f"Time range: {gdf_crash['created_time'].min()} to {gdf_crash['created_time'].max()}")
gdf_crash.head()

Processing with STM-Graph...
Running in testing mode with bounds: {'min_lat': 40.4774, 'max_lat': 40.9176, 'min_lon': -74.2591, 'max_lon': -73.7004}
Reading dataset from nyc_crash_311/temp_combined_crash.csv...
Reading CSV file: nyc_crash_311/temp_combined_crash.csv
Original dataset shape: (636910, 30)
Filtered by bounds: 594655 → 588183 rows
Sorting dataset by timestamp (ascending order)...
Final dataset shape: (588183, 31)
Time range: 2020-01-02 00:00:00 to 2026-06-11 13:08:00
Generating visualizations in nyc_crash_311/stm_graph/nyc/crash/preprocess...
Spatial visualization saved to nyc_crash_311/stm_graph/nyc/crash/preprocess/spatial_distribution.png
Temporal visualization saved to nyc_crash_311/stm_graph/nyc/crash/preprocess/temporal_distribution.png
Processed dataset shape: (588183, 31)
Time range: 2020-01-02 00:00:00 to 2026-06-11 13:08:00


,CRASH DATE,CRASH TIME,borough,zip_code,latitude,longitude,location,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,collision_id,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5,created_time,geometry
304,01/02/2020,0:00,NaN,NaN,40.586117,-73.936040,"(40.586117, -73.93604)",BELT PARKWAY,NaN,NaN,...,NaN,NaN,4268205,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,2020-01-02,POINT (-73.93604 40.58612)
409,01/02/2020,0:00,QUEENS,11106.0,40.768414,-73.935410,"(40.768414, -73.93541)",NaN,NaN,11-08 31 DRIVE,...,NaN,NaN,4268171,Sedan,Sedan,NaN,NaN,NaN,2020-01-02,POINT (-73.93541 40.76841)
392,01/02/2020,0:00,BROOKLYN,11232.0,40.656372,-74.009384,"(40.656372, -74.009384)",37 STREET,2 AVENUE,NaN,...,NaN,NaN,4268293,Sedan,Sedan,NaN,NaN,NaN,2020-01-02,POINT (-74.00938 40.65637)
106,01/02/2020,0:00,MANHATTAN,10024.0,40.785732,-73.976450,"(40.785732, -73.97645)",WEST 83 STREET,AMSTERDAM AVENUE,NaN,...,NaN,NaN,4268268,Sedan,Sedan,NaN,NaN,NaN,2020-01-02,POINT (-73.97645 40.78573)
323,01/02/2020,0:00,NaN,NaN,40.734480,-73.922430,"(40.73448, -73.92243)",BROOKLYN QUEENS EXPRESSWAY,NaN,NaN,...,NaN,NaN,4268106,Taxi,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,2020-01-02,POINT (-73.92243 40.73448)


In [5]:
# Apply grid mapping to the data
print("Applying spatial mapping...")

# Create a grid mapping with 1000-meter cells
mapper = stm_graph.GridMapping(cell_size=1000.0, target_crs="EPSG:32618")

# Apply the mapping to get district geometries and point-to-partition mapping
district_gdf, point_to_partition = mapper.create_mapping(gdf_crash)

print(f"Created mapping with {len(district_gdf)} regions")
print(f"Points with valid mapping: {(point_to_partition >= 0).sum()} of {len(point_to_partition)}")

# Visualize the mapping
mapper.visualize(
    points_gdf=gdf_crash,
    partition_gdf=district_gdf,
    point_to_partition=point_to_partition,
    out_dir=OUTPUT_DIR,
    file_format="png"
)

district_gdf.head()

Applying spatial mapping...


/Users/jacobboyar/Library/Mobile Documents/.Trash/Group-7-Capstone/model/stm_graph/stm_graph/.venv/lib/python3.8/site-packages/pandas/core/reshape/merge.py:1203: RuntimeWarning: invalid value encountered in cast
  if not (lk == lk.astype(rk.dtype))[~np.isnan(lk)].all():


Created mapping with 2162 regions
Points with valid mapping: 588128 of 588183
Grid mapping visualizations saved to nyc_crash_311/stm_graph/nyc/crash/mapping in PNG format!


,geometry
0,"POLYGON ((563122.146 4483411.057, 564122.146 4..."
1,"POLYGON ((563122.146 4484411.057, 564122.146 4..."
2,"POLYGON ((563122.146 4485411.057, 564122.146 4..."
3,"POLYGON ((563122.146 4486411.057, 564122.146 4..."
4,"POLYGON ((563122.146 4487411.057, 564122.146 4..."


In [6]:
# Define the feature types to extract
feature_types = ['poi', 'road', 'junction']

# Extract OSM features
osm_cache_dir = os.path.join(OUTPUT_DIR, "osm_cache")
osm_features = stm_graph.extract_osm_features(
    regions_gdf=district_gdf,
    bounds=NYC_BOUNDS,
    cache_dir=osm_cache_dir,
    feature_types=feature_types,
    normalize=True,
    meter_crs="EPSG:32618",
    lat_lon_crs="EPSG:4326"
)

# Print available features
print(f"Extracted {len(osm_features.columns)} OSM features")
print("\nFeature sample:")
osm_features.head()

Extracting OpenStreetMap features...
Loading OSM data from cache: nyc_crash_311/stm_graph/nyc/crash/osm_cache/osm_data_40p91760_40p47740_-73p70040_-74p25910.pkl
Successfully loaded OSM data from cache
Calculating POI densities...


100%|██████████| 9/9 [04:00<00:00, 26.73s/it] 


Calculating junction counts...
Calculating road density and features...


100%|██████████| 2162/2162 [01:00<00:00, 35.72it/s]


Extracted 14 OSM features for 2162 regions
Extracted 14 OSM features

Feature sample:


,poi_shop_density,poi_healthcare_density,poi_service_density,poi_entertainment_density,poi_religious_density,poi_food_density,poi_education_density,poi_transportation_density,poi_recreation_density,poi_total_density,junction_density,road_length_m,avg_speed_kmh,road_density_m_per_m2
0,0.000615,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000003,0.0,0.000618,0.000048,20501.893781,33.531387,0.020502
1,0.000526,0.0,0.000002,0.0,0.000006,0.000004,0.000002,0.000013,0.0,0.000553,0.000055,16391.739051,33.531387,0.016392
2,0.000001,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000001,0.000000,2006.466045,45.000000,0.002006
3,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,33.531387,0.000000
4,0.000001,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000001,0.000000,617.490900,33.531387,0.000617


In [7]:
# Filter points that have valid mappings
gdf_crash_valid = gdf_crash[point_to_partition >= 0].copy()
point_to_partition_valid = point_to_partition[point_to_partition >= 0].copy()

print(f"Using {len(gdf_crash_valid)} valid points for graph construction")

# Build graph with static features
graph_data = stm_graph.build_graph_and_augment(
    grid_gdf=district_gdf,
    points_gdf=gdf_crash_valid,
    point_to_cell=point_to_partition_valid,
    adj_matrix=None,
    remove_empty_nodes=True,
    out_dir=OUTPUT_DIR,
    save_flag=True,
    static_features=osm_features,
)

# Extract graph components
edge_index = graph_data["edge_index"]
edge_weight = graph_data["edge_weight"]
node_features = graph_data["node_features"]
augmented_df = graph_data["augmented_df"]
node_ids = graph_data["node_ids"]

print(f"Built graph with {edge_index.shape[1]} edges and {graph_data['num_nodes']} nodes")
print(f"Node features shape: {node_features.shape}")

# Display augmented dataframe
augmented_df.head()

Using 588128 valid points for graph construction
Removing empty regions...
Before: 2162 regions
After: 881 regions
Saving graph components to nyc_crash_311/stm_graph/nyc/crash...
Built graph with 6270 edges and 881 nodes
Node features shape: (881, 15)


,CRASH DATE,CRASH TIME,borough,zip_code,latitude,longitude,location,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 5,collision_id,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5,created_time,geometry,cell_id
304,01/02/2020,0:00,NaN,NaN,40.586117,-73.936040,"(40.586117, -73.93604)",BELT PARKWAY,NaN,NaN,...,NaN,4268205,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,2020-01-02,POINT (-73.93604 40.58612),341
409,01/02/2020,0:00,QUEENS,11106.0,40.768414,-73.935410,"(40.768414, -73.93541)",NaN,NaN,11-08 31 DRIVE,...,NaN,4268171,Sedan,Sedan,NaN,NaN,NaN,2020-01-02,POINT (-73.93541 40.76841),362
392,01/02/2020,0:00,BROOKLYN,11232.0,40.656372,-74.009384,"(40.656372, -74.009384)",37 STREET,2 AVENUE,NaN,...,NaN,4268293,Sedan,Sedan,NaN,NaN,NaN,2020-01-02,POINT (-74.00938 40.65637),191
106,01/02/2020,0:00,MANHATTAN,10024.0,40.785732,-73.976450,"(40.785732, -73.97645)",WEST 83 STREET,AMSTERDAM AVENUE,NaN,...,NaN,4268268,Sedan,Sedan,NaN,NaN,NaN,2020-01-02,POINT (-73.97645 40.78573),276
323,01/02/2020,0:00,NaN,NaN,40.734480,-73.922430,"(40.73448, -73.92243)",BROOKLYN QUEENS EXPRESSWAY,NaN,NaN,...,NaN,4268106,Taxi,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,2020-01-02,POINT (-73.92243 40.73448),392


In [8]:
# Create temporal dataset
temporal_dataset, dataset_path, metadata = stm_graph.create_temporal_dataset(
    edge_index=edge_index,
    augmented_df=augmented_df,
    edge_weights=edge_weight,
    node_ids=node_ids,
    static_features=osm_features,
    time_col="created_time",
    cell_col="cell_id",
    bin_type="daily",
    interval_hours=1,
    history_window=3,
    use_time_features=False,
    task="classification",
    horizon=1,
    downsample_factor=1,
    normalize=True,
    scaler_type="minmax",
    dataset_name="nyc_crash_dataset",
    output_format="4d",
)

Using daily time bins
Time features disabled: using only count features
Feature dimensions:
- Total timesteps: 2353
- Number of nodes: 881
- History window: 3 steps
- Base features per step: 1
- Shape: [timesteps, nodes, history, features] = (2353, 881, 3, 1)
- Time span: 2352 days 00:00:00
- Bin type: daily

Feature structure for each history step (h):
- Feature 0: Event counts
Integrating static features into 4D temporal features...
Added 14 static features to each time step and history window
New features shape: (2353, 881, 3, 15)
Scaling 4D features using minmax scaling...
Converted to binary classification task: 5.13% positive samples
Prepared 4D forecasting data: Input shape: (2352, 881, 3, 15), Target shape: (2352, 881, 1)
Created temporal graph dataset:
- Timesteps: 2352
- Feature shape: torch.Size([881, 3, 15])
- Edges: 6270


In [ ]:
temporal_dataset_3d = stm_graph.convert_4d_to_3d_dataset(
    temporal_dataset, static_features_count=osm_features.shape[1])

# Plot time series for the most active nodes
stm_graph.plot_node_time_series(
    temporal_dataset_3d,
    num_nodes=5,  # Show 5 nodes
    selection_method="highest_activity",  # Select most active nodes
    feature_idx=0,  # Event count feature
    plot_type="2d",  # 2D line plot
    start_time="2020-01-01",  # Start date for x-axis
    time_delta=timedelta(hours=1),  # Hourly data
    title="Crash Events Over Time (Most Active Nodes)",
    figsize=(15, 8),
    out_dir=OUTPUT_DIR,
    filename="time_series_top",
    file_format="png",
)

Converted dataset from 4D to 3D format
Original 4D shape: [T=2352, N=881, H=3, F=15]
New 3D shape: [T=2352, N=881, F=17]
Static features: 14
Dynamic features per history step: 1
Selected 5 most active nodes: [89, 41, 68, 43, 40]
Time series plot saved to nyc_crash_311/stm_graph/nyc/crash/graph/time_series_top.png


'nyc_crash_311/stm_graph/nyc/crash/graph/time_series_top.png'

In [10]:
# Plot 3D visualization for most active nodes
stm_graph.plot_node_time_series(
    temporal_dataset_3d,
    num_nodes=3,  # Show 3 nodes
    selection_method="highest_activity",  # Select most active nodes
    feature_idx=0,  # Event count feature
    plot_type="3d",  # 3D surface plot
    n_steps=168,  # First week (7 days * 24 hours)
    title="Crashes 3D Visualization Over Time",
    figsize=(15, 10),
    out_dir=OUTPUT_DIR,
    filename="time_series_3d",
    file_format="png",
)

Selected 3 most active nodes: [68, 43, 40]
Time series plot saved to nyc_crash_311/stm_graph/nyc/crash/graph/time_series_3d.png


'nyc_crash_311/stm_graph/nyc/crash/graph/time_series_3d.png'

In [11]:
# Extract event counts for each region (node) at a specific time
time_step = 24  # Example: events after 24 hours
node_counts = np.array(
    [
        temporal_dataset_3d.features[time_step][node, 0].item()
        for node in range(graph_data["num_nodes"])
    ]
)

# Plot spatial network with region and edge colors
stm_graph.plot_spatial_network(
    regions_gdf=district_gdf,
    edge_index=edge_index,
    edge_weights=edge_weight,
    node_values=node_counts,
    node_ids=node_ids,
    time_step=time_step,
    title="Crash Density After 24 Hours",
    node_cmap="YlOrRd",  # Red-yellow colormap for heat
    edge_cmap="viridis",  # Blue-green for edges
    map_style=ctx.providers.CartoDB.Positron,
    figsize=(15, 15),
    out_dir=OUTPUT_DIR,
    filename="spatial_network",
    file_format="png",
)

Filtered from 2162 to 881 active grid cells
All edge weights are identical: 1.0000
Spatial network plot saved to nyc_crash_311/stm_graph/nyc/crash/graph/spatial_network.png


'nyc_crash_311/stm_graph/nyc/crash/graph/spatial_network.png'

In [ ]:
# Create a temporal heatmap to see patterns across time and nodes
stm_graph.plot_temporal_heatmap(
    temporal_dataset_3d,
    num_nodes=10,
    feature_idx=0,  # Event count feature
    selection_method="highest_activity",
    start_time="2020-01-01",
    time_delta=timedelta(hours=1),
    n_steps=168,  # First week
    title="Crash Events Temporal Heatmap (First Week)",
    figsize=(15, 8),
    out_dir=OUTPUT_DIR,
    filename="temporal_heatmap",
    file_format="png",
)

Selected 10 most active nodes: [70, 92, 45, 84, 66, 89, 41, 68, 43, 40]
Temporal heatmap saved to nyc_crash_311/stm_graph/nyc/crash/graph/temporal_heatmap.png


'nyc_crash_311/stm_graph/nyc/crash/graph/temporal_heatmap.png'

In [13]:
# Discover available model options
print("Available models in STM-Graph:")
stm_graph.list_available_models()

Available models in STM-Graph:
Available custom models:
- agcrn: Adaptive Graph Convolutional Recurrent Network with node-specific parameters
- dcrnn: Diffusion Convolutional Recurrent Neural Network for traffic forecasting
- gcn: Graph Convolutional Network for simple graph learning tasks
- stgcn: Spatio-Temporal Graph Convolutional Network with temporal gating
- tgcn: Temporal Graph Convolutional Network with recurrent architecture

For detailed information about a model, use get_model_info(model_name)

To list PyTorch Geometric Temporal models, use list_pytorch_geometric_temporal_models()


In [ ]:
# # STGCN
# model = stm_graph.create_model(
#     model_name="stgcn",
#     source="custom",
#     num_nodes=temporal_dataset.features[0].shape[0],
#     in_channels=temporal_dataset.features[0].shape[2],
#     out_channels=1,
#     hidden_dim=64,
#     k=3,
#     embedding_dimensions=16,
#     dropout=0.2,
#     task="classification",
# )

# # Train the model
# results = stm_graph.train_model(
#     model=model,
#     dataset=temporal_dataset,
#     optimizer_name="adam",
#     learning_rate=0.0001,
#     task="classification",
#     num_epochs=500,  
#     batch_size=10,
#     batch_to_device=True,
#     test_size=0.15,
#     val_size=0.15,
#     use_nested_tqdm=True,
#     early_stopping=True,
#     patience=50,
#     scheduler_type="step",
#     lr_decay_epochs=50,
#     lr_decay_factor=1,
#     wandb_project="stm_graph_crash",
#     experiment_name="stgcn",
#     use_wandb=True, 
#     fixed_batch_size=True,
#     log_dir=OUTPUT_DIR,
# )

2026-10-04 23:28:09,577 - INFO - ==================== EXPERIMENT: stgcn ====================
2026-10-04 23:28:09,578 - INFO - Started training at: 20261004_232809
2026-10-04 23:28:09,579 - INFO - Model: ClassificationWrapper
2026-10-04 23:28:09,580 - INFO - Task: classification
2026-10-04 23:28:09,581 - INFO - Batch size: 10 (fixed: True)
2026-10-04 23:28:09,582 - INFO - Learning rate: 0.0001
2026-10-04 23:28:09,583 - INFO - Epochs: 500
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: